# ACT (Action Chunking with Transformers) - PushT Training

This notebook trains an ACT policy on the custom PushT dataset with WandB logging.
It includes standalone inference cells that can be run during or after training.

**Run this on a machine with a GPU.** Cell 0 handles cloning the repo and installing dependencies.

**Architecture**: CVAE + Transformer (encoder-decoder) with ResNet18 vision backbone

**Key differences from Diffusion Policy**:
- ACT uses a VAE to model action distribution, DP uses iterative denoising
- ACT predicts full action chunks in one forward pass, DP refines over multiple diffusion steps
- ACT trains with L1 reconstruction + KL divergence loss, DP trains with MSE denoising loss

## Cell 0: Clone Repo & Install Dependencies (Run this first on a new machine)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/SBprjcts/DiffusionPolicyTraining.git"
BRANCH = "feat/act-implementation"
REPO_DIR = "DiffusionPolicyTraining"

# Clone the repo if not already present
if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
else:
    print(f"Repo already exists at ./{REPO_DIR}, pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Change working directory into the repo
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Install lerobot with pusht environment support + wandb
subprocess.run(["pip", "install", "-e", ".[pusht]"], check=True)
subprocess.run(["pip", "install", "wandb", "imageio[ffmpeg]"], check=True)

print("\nSetup complete!")

## Cell 1: Setup & Imports

In [ ]:
# ============================================================
# USER CONFIG - Edit these values
# ============================================================
HF_USERNAME = "SaifB"  # Replace with your HuggingFace username

DATASET_PATH = "custom_pusht_data/custom_pusht"  # Local custom dataset
OUTPUT_DIR = Path("outputs/act_checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
BATCH_SIZE = 8
TRAINING_STEPS = 100_000
SAVE_EVERY = 5_000
LOG_EVERY = 100
EVAL_EVERY = 10_000  # Run evaluation during training every N steps

# WandB config
WANDB_PROJECT = "lerobot-act-pusht"
WANDB_ENABLED = True

print(f"Output directory: {OUTPUT_DIR}")
print(f"Dataset: {DATASET_PATH}")
print(f"Training steps: {TRAINING_STEPS}")
print(f"WandB: {'enabled' if WANDB_ENABLED else 'disabled'}")

## Cell 3: Load Dataset

In [ ]:
from lerobot.configs.types import FeatureType
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig

# Load dataset metadata to get features and stats
dataset_metadata = LeRobotDatasetMetadata(DATASET_PATH)
features = dataset_to_policy_features(dataset_metadata.features)

print("Dataset features:")
for name, feat in features.items():
    print(f"  {name}: type={feat.type}, shape={feat.shape}")

# Split into input/output features
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}

print(f"\nInput features: {list(input_features.keys())}")
print(f"Output features: {list(output_features.keys())}")
print(f"FPS: {dataset_metadata.fps}")

## Cell 4: Build ACT Policy

In [ ]:
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors


def make_delta_timestamps(delta_indices, fps):
    if delta_indices is None:
        return [0]
    return [i / fps for i in delta_indices]


# Create ACT config
cfg = ACTConfig(
    input_features=input_features,
    output_features=output_features,
    chunk_size=100,
    n_action_steps=100,
    use_vae=True,
    latent_dim=32,
    dim_model=512,
    n_heads=8,
    n_encoder_layers=4,
    n_decoder_layers=1,
    vision_backbone="resnet18",
    pretrained_backbone_weights="ResNet18_Weights.IMAGENET1K_V1",
    kl_weight=10.0,
    dropout=0.1,
)

# Create policy
policy = ACTPolicy(cfg)
policy.train()
policy.to(device)

# Create pre/post processors for normalization
preprocessor, postprocessor = make_pre_post_processors(cfg, dataset_stats=dataset_metadata.stats)

# Compute delta timestamps for action chunking
delta_timestamps = {
    "action": make_delta_timestamps(cfg.action_delta_indices, dataset_metadata.fps),
}
delta_timestamps |= {
    k: make_delta_timestamps(cfg.observation_delta_indices, dataset_metadata.fps)
    for k in cfg.image_features
}

# Load full dataset with delta timestamps
dataset = LeRobotDataset(DATASET_PATH, delta_timestamps=delta_timestamps)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=device.type != "cpu",
    drop_last=True,
    num_workers=2,
)

# Optimizer (using ACT defaults)
optimizer = cfg.get_optimizer_preset().build(policy.parameters())

num_params = sum(p.numel() for p in policy.parameters())
num_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"ACT Policy created")
print(f"  Total parameters: {num_params:,}")
print(f"  Trainable parameters: {num_trainable:,}")
print(f"  Chunk size: {cfg.chunk_size}")
print(f"  VAE: {cfg.use_vae} (latent_dim={cfg.latent_dim})")
print(f"  Dataset size: {len(dataset)} frames")
print(f"  Batch size: {BATCH_SIZE}")

## Cell 5: Training Loop with WandB

In [ ]:
import wandb

# Initialize WandB
if WANDB_ENABLED:
    wandb.init(
        project=WANDB_PROJECT,
        config={
            "policy": "ACT",
            "dataset": DATASET_PATH,
            "batch_size": BATCH_SIZE,
            "training_steps": TRAINING_STEPS,
            "chunk_size": cfg.chunk_size,
            "use_vae": cfg.use_vae,
            "latent_dim": cfg.latent_dim,
            "dim_model": cfg.dim_model,
            "n_heads": cfg.n_heads,
            "n_encoder_layers": cfg.n_encoder_layers,
            "n_decoder_layers": cfg.n_decoder_layers,
            "kl_weight": cfg.kl_weight,
            "lr": cfg.optimizer_lr,
            "device": str(device),
        },
        name=f"act_pusht_{datetime.now():%Y%m%d_%H%M%S}",
    )

print("Starting training...")
policy.train()

step = 0
done = False
losses = []

while not done:
    for batch in dataloader:
        batch = preprocessor(batch)
        loss, loss_dict = policy.forward(batch)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)

        optimizer.step()
        optimizer.zero_grad()

        loss_val = loss.item()
        losses.append(loss_val)

        # Logging
        if step % LOG_EVERY == 0:
            avg_loss = np.mean(losses[-LOG_EVERY:]) if losses else loss_val
            log_msg = f"step: {step:>6d}/{TRAINING_STEPS} | loss: {loss_val:.4f} | avg_loss: {avg_loss:.4f}"
            if loss_dict:
                for k, v in loss_dict.items():
                    log_msg += f" | {k}: {v:.4f}"
            print(log_msg)

            if WANDB_ENABLED:
                wandb_log = {"loss": loss_val, "avg_loss": avg_loss, "step": step}
                if loss_dict:
                    wandb_log.update({k: v for k, v in loss_dict.items()})
                wandb.log(wandb_log, step=step)

        # Save checkpoint
        if step > 0 and step % SAVE_EVERY == 0:
            ckpt_dir = OUTPUT_DIR / f"checkpoint_{step:06d}"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            policy.save_pretrained(ckpt_dir)
            preprocessor.save_pretrained(ckpt_dir)
            postprocessor.save_pretrained(ckpt_dir)
            print(f"Checkpoint saved to {ckpt_dir}")

        step += 1
        if step >= TRAINING_STEPS:
            done = True
            break

# Save final model
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)
policy.save_pretrained(final_dir)
preprocessor.save_pretrained(final_dir)
postprocessor.save_pretrained(final_dir)
print(f"\nTraining complete! Final model saved to {final_dir}")
print(f"Final loss: {losses[-1]:.4f}")

if WANDB_ENABLED:
    wandb.finish()

## Cell 6: Inference / Evaluation (Standalone)

This cell can be run **during training** (loading a checkpoint) or **after training** (loading the final model).
It creates a PushT environment, runs the policy, records a video, and reports success rate.

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from pathlib import Path
from IPython.display import Video, display

from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.envs.configs import PushtEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.scripts.lerobot_eval import rollout
from lerobot.utils.io_utils import write_video

# ============================================================
# CONFIGURE: Point to the checkpoint you want to evaluate
# ============================================================
# During training, use a checkpoint:
#   EVAL_MODEL_PATH = "outputs/act_checkpoints/checkpoint_005000"
# After training, use the final model:
EVAL_MODEL_PATH = "outputs/act_checkpoints/final_model"

N_EVAL_EPISODES = 5
EVAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VIDEOS_DIR = Path("outputs/act_checkpoints/eval_videos")
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Load policy from checkpoint
print(f"Loading policy from: {EVAL_MODEL_PATH}")
eval_cfg = ACTConfig.from_pretrained(EVAL_MODEL_PATH)
eval_cfg.device = EVAL_DEVICE
eval_policy = ACTPolicy.from_pretrained(EVAL_MODEL_PATH, config=eval_cfg)
eval_policy.eval()
eval_policy.to(EVAL_DEVICE)

# Load processors from checkpoint
eval_preprocessor, eval_postprocessor = make_pre_post_processors(
    eval_cfg, pretrained_path=EVAL_MODEL_PATH
)

# Create PushT environment
env_cfg = PushtEnv()
env_dict = make_env(env_cfg, n_envs=1)
env = env_dict["pusht"][0]

# Create env-level processors (identity for PushT)
env_pre, env_post = make_env_pre_post_processors(env_cfg, eval_cfg)

print(f"Running {N_EVAL_EPISODES} evaluation episodes...")
successes = []
rewards = []

for ep_idx in range(N_EVAL_EPISODES):
    # Collect frames for video
    ep_frames = []

    def render_callback(env):
        if isinstance(env, gym.vector.SyncVectorEnv):
            ep_frames.append(env.envs[0].render())

    result = rollout(
        env=env,
        policy=eval_policy,
        env_preprocessor=env_pre,
        env_postprocessor=env_post,
        preprocessor=eval_preprocessor,
        postprocessor=eval_postprocessor,
        seeds=[ep_idx * 1000],
        render_callback=render_callback,
    )

    ep_success = result["success"][0].any().item()
    ep_reward = result["reward"][0].sum().item()
    successes.append(ep_success)
    rewards.append(ep_reward)
    print(f"  Episode {ep_idx}: reward={ep_reward:.2f}, success={ep_success}")

    # Save video for the first episode
    if ep_idx == 0 and ep_frames:
        video_path = VIDEOS_DIR / f"eval_episode_{ep_idx}.mp4"
        frames_tensor = torch.from_numpy(np.stack(ep_frames))
        write_video(str(video_path), frames_tensor, fps=env_cfg.fps)
        print(f"  Video saved to {video_path}")

env.close()

success_rate = np.mean(successes) * 100
avg_reward = np.mean(rewards)
print(f"\nResults ({N_EVAL_EPISODES} episodes):")
print(f"  Success rate: {success_rate:.1f}%")
print(f"  Average reward: {avg_reward:.2f}")

# Display video inline
video_file = VIDEOS_DIR / "eval_episode_0.mp4"
if video_file.exists():
    display(Video(str(video_file), embed=True, width=384))

## Cell 7: Push to HuggingFace Hub

Push the trained model and dataset to your HF profile.

In [ ]:
from huggingface_hub import login

# Login to HuggingFace (will prompt for token)
login()

# Make sure HF_USERNAME is set
assert HF_USERNAME != "<your-username>", "Please set HF_USERNAME in Cell 2!"

MODEL_PATH = "outputs/act_checkpoints/final_model"

# Load the final model
push_cfg = ACTConfig.from_pretrained(MODEL_PATH)
push_policy = ACTPolicy.from_pretrained(MODEL_PATH, config=push_cfg)
push_pre, push_post = make_pre_post_processors(push_cfg, pretrained_path=MODEL_PATH)

# Push model to Hub
hub_repo = f"{HF_USERNAME}/act_pusht"
print(f"Pushing model to {hub_repo}...")
push_policy.push_to_hub(hub_repo)
push_pre.push_to_hub(hub_repo)
push_post.push_to_hub(hub_repo)
print(f"Model pushed to https://huggingface.co/{hub_repo}")

# Push dataset to Hub
dataset_repo = f"{HF_USERNAME}/custom_pusht"
print(f"\nPushing dataset to {dataset_repo}...")
push_dataset = LeRobotDataset(DATASET_PATH)
push_dataset.push_to_hub(repo_id=dataset_repo)
print(f"Dataset pushed to https://huggingface.co/datasets/{dataset_repo}")